# 01 — Download audio da Xeno-canto

Questo notebook mostra come usare `src/download.py` per scaricare tracce audio `.mp3` dall'API [Xeno-canto](https://xeno-canto.org/) e organizzarle in `data/raw/<specie>/`.

**Pipeline:**
1. Setup ambiente (Colab / locale)
2. Configurazione API key e lista specie
3. Ricerca registrazioni
4. Download audio
5. Verifica dei file scaricati

## 1. Setup ambiente

Il blocco seguente rileva automaticamente se siamo in Google Colab e, in caso affermativo, clona la repo e installa le dipendenze.

In [ ]:
import sys
import os

# Detect Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Google Drive (optional — for persistent storage)
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone the repository if not already present
    repo_dir = '/content/bird-acoustics-classifier'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/danort92/bird-acoustics-classifier.git {repo_dir}

    os.chdir(repo_dir)
    !pip install -q -r requirements.txt
    print('Colab setup complete.')
else:
    # Local: ensure we are at the project root
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(project_root)
    print(f'Working directory: {os.getcwd()}')

## 2. Configurazione

Imposta l'**API key** di Xeno-canto e la **lista delle specie** da scaricare.

> **API key**: opzionale per l'endpoint pubblico v2.  Se hai una chiave registrata,
> impostala come variabile d'ambiente `XENO_CANTO_API_KEY` oppure inseriscila
> quando richiesta dal modulo.

In [ ]:
# Opzione A — variabile d'ambiente (consigliata)
# os.environ['XENO_CANTO_API_KEY'] = 'la_tua_chiave'

# Lista specie (nomi scientifici)
SPECIES = [
    "Turdus merula",       # Merlo
    "Erithacus rubecula",  # Pettirosso
    "Parus major",         # Cinciallegra
]

MAX_PER_SPECIES = 30   # Massimo numero di file per specie
QUALITY = "A"          # Qualità minima (A = migliore, E = peggiore)
OUTPUT_DIR = "data/raw"

print(f"Specie selezionate: {SPECIES}")
print(f"Max registrazioni/specie: {MAX_PER_SPECIES}")
print(f"Qualità minima: {QUALITY}")
print(f"Directory output: {OUTPUT_DIR}")

## 3. Inizializzazione del downloader

In [ ]:
from src.download import XenoCantoDownloader

# Il costruttore legge automaticamente XENO_CANTO_API_KEY
# o chiede la chiave in modo interattivo se non impostata.
downloader = XenoCantoDownloader(
    output_dir=OUTPUT_DIR,
    request_delay=0.5,  # pausa tra le richieste (secondi)
)
print("Downloader inizializzato.")

## 4. Ricerca registrazioni (preview senza download)

Prima di scaricare, verifichiamo quante registrazioni sono disponibili per ogni specie.

In [ ]:
import pandas as pd

preview_rows = []

for species in SPECIES:
    recordings = downloader.search_species(species, quality=QUALITY, max_results=MAX_PER_SPECIES)
    for r in recordings[:3]:  # mostra solo i primi 3 per brevità
        preview_rows.append({
            "species": species,
            "id": r.get("id"),
            "country": r.get("cnt"),
            "quality": r.get("q"),
            "length": r.get("length"),
            "file_url": r.get("file"),
        })
    print(f"{species}: {len(recordings)} registrazioni trovate")

df_preview = pd.DataFrame(preview_rows)
df_preview

## 5. Download audio

Scarica tutte le registrazioni per le specie configurate e le salva in `data/raw/<specie>/`.

In [ ]:
results = downloader.download_species(
    species_list=SPECIES,
    max_per_species=MAX_PER_SPECIES,
    quality=QUALITY,
)

print("\n=== Riepilogo download ===")
for species, paths in results.items():
    print(f"  {species}: {len(paths)} file scaricati")

## 6. Verifica file scaricati

In [ ]:
from pathlib import Path

downloaded = downloader.list_downloaded()

summary_rows = []
for species_dir, files in downloaded.items():
    total_mb = sum(f.stat().st_size for f in files) / 1_048_576
    summary_rows.append({
        "species_dir": species_dir,
        "n_files": len(files),
        "total_MB": round(total_mb, 2),
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
print(f"\nTotale file: {df_summary['n_files'].sum()}")
print(f"Totale dimensione: {df_summary['total_MB'].sum():.2f} MB")

## 7. Struttura directory risultante

Visualizza l'albero delle directory dopo il download.

In [ ]:
def print_tree(root: Path, max_files: int = 5) -> None:
    """Print a compact directory tree."""
    print(f"{root}/")
    for species_dir in sorted(root.iterdir()):
        if not species_dir.is_dir():
            continue
        files = sorted(species_dir.glob('*.mp3'))
        print(f"  {species_dir.name}/  ({len(files)} file)")
        for f in files[:max_files]:
            print(f"    {f.name}")
        if len(files) > max_files:
            print(f"    … e altri {len(files) - max_files} file")

print_tree(Path(OUTPUT_DIR))

---
**Prossimo step:** `02_preprocessing.ipynb` — conversione audio → spettrogrammi mel